# Task 5 — Pivot Translation (Italian → Swedish via English)

**Project 2.2 · Vasilis & Riccardo**

A *pivot* language is an intermediary: to translate A→B without A–B training data,
translate A→P and then P→B. Here **A = Italian, P = English, B = Swedish**.

**Why this is worth doing.** Europarl gives us plenty of Italian–English and
Swedish–English data, but no Italian–Swedish corpus. Pivoting lets us build an IT→SV
system from the resources we actually have. The cost is **error propagation**: whatever
the IT→EN system gets wrong is fed to the EN→SV system as if it were correct, and cannot
be recovered.

**The evaluation problem.** To measure IT→SV we still need genuine Italian–Swedish
reference pairs. The `it-en` and `sv-en` corpora are aligned *independently*, so line *n*
of one is not the same sentence as line *n* of the other. We build the missing corpus by
joining the two on their **shared English side** — both are transcripts of the same
parliamentary proceedings, so most English sentences occur verbatim in both. This yields
**59,524 IT–EN–SV triples** (all sides ≤ 20 tokens), prepared by
`scripts/build_pivot_triples.py` and committed to the repo.

**What we compare.** Three systems, all trained on the same triples and evaluated on the
same held-out test split:

| system | path |
|---|---|
| IT→EN | first leg, measured on its own |
| EN→SV | second leg, measured on its own |
| **pivot IT→EN→SV** | the two legs chained |
| **direct IT→SV** | trained on the IT–SV pairs directly |

The direct system is the upper bound the pivot is trying to reach: it sees the same data
but never passes through English. The gap between them *is* the cost of pivoting.

All models use the **attention** architecture from Task 4, which we showed clearly
outperforms the fixed-context encoder–decoder.

In [ ]:
# --- Colab setup (skipped when running locally) ---
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/riccardotella/English-Italian-Machine-Translation-.git"

if "google.colab" in sys.modules:
    if not Path("English-Italian-Machine-Translation-").exists():
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir("English-Italian-Machine-Translation-")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)

print("cwd:", os.getcwd())

In [ ]:
# --- Imports, seeds, device ---
import random, json, gzip, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PIVOT_DIR   = PROJECT_ROOT / "data" / "pivot"
MODEL_DIR   = PROJECT_ROOT / "models" / "task5"
RESULTS_DIR = PROJECT_ROOT / "results" / "task5_outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")

## 1. The IT–EN–SV triples

Built by `scripts/build_pivot_triples.py` and committed gzipped (4.4 MB total), so this
notebook runs from a fresh clone without downloading the 560 MB Swedish corpus.

In [ ]:
def read_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

italian  = read_gz(PIVOT_DIR / "triples.it.gz")
english  = read_gz(PIVOT_DIR / "triples.en.gz")
swedish  = read_gz(PIVOT_DIR / "triples.sv.gz")
assert len(italian) == len(english) == len(swedish), "Triples are not aligned!"

triples = list(zip(italian, english, swedish))
print(f"IT-EN-SV triples: {len(triples):,}")
print("\nexample:")
for tag, s in zip(("IT", "EN", "SV"), triples[0]):
    print(f"  {tag}: {s}")

In [ ]:
# --- 70/10/20 split, same protocol and seed as Tasks 3-4 ---
TEST_RATIO, VALIDATION_RATIO = 0.20, 0.10

n = len(triples)
n_test  = int(n * TEST_RATIO)
n_val   = int(n * VALIDATION_RATIO)
n_train = n - n_val - n_test

rng = np.random.default_rng(SEED)
idx = rng.permutation(n)
train_triples = [triples[i] for i in idx[:n_train]]
val_triples   = [triples[i] for i in idx[n_train:n_train + n_val]]
test_triples  = [triples[i] for i in idx[n_train + n_val:]]

print(f"train {len(train_triples):,} | val {len(val_triples):,} | test {len(test_triples):,}")

# Each system is a (source, target) view of the SAME split, so every comparison
# below is on identical sentences.
def view(triples_subset, source_index, target_index):
    return [(t[source_index], t[target_index]) for t in triples_subset]

IT, EN, SV = 0, 1, 2

In [ ]:
# --- Vocabularies (built on the training split only) ---
MAX_VOCABULARY_SIZE = 10_000
SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>"]
PAD_INDEX, UNK_INDEX, SOS_INDEX, EOS_INDEX = range(len(SPECIAL_TOKENS))

def tokenize(sentence):
    return sentence.split()

def build_vocabulary(sentences, max_size=MAX_VOCABULARY_SIZE):
    counts = Counter(w for s in sentences for w in tokenize(s))
    words = [w for w, _ in counts.most_common(max_size - len(SPECIAL_TOKENS))]
    index_to_token = SPECIAL_TOKENS + words
    return {t: i for i, t in enumerate(index_to_token)}, index_to_token

vocabularies = {}
for name, position in (("it", IT), ("en", EN), ("sv", SV)):
    token_to_index, index_to_token = build_vocabulary(
        [t[position] for t in train_triples])
    vocabularies[name] = (token_to_index, index_to_token)
    print(f"{name} vocabulary: {len(token_to_index):,}")

# Out-of-vocabulary rate per language: relevant because Task 1 predicted that
# morphologically richer targets are harder to generate.
for name, position in (("it", IT), ("en", EN), ("sv", SV)):
    token_to_index, _ = vocabularies[name]
    tokens = [w for t in train_triples for w in tokenize(t[position])]
    oov = sum(1 for w in tokens if w not in token_to_index)
    print(f"  {name} OOV rate: {100 * oov / len(tokens):.2f}%")

In [ ]:
# --- Dataset / loaders ---
BATCH_SIZE = 128

def encode_sentence(sentence, token_to_index):
    ids = [token_to_index.get(w, UNK_INDEX) for w in tokenize(sentence)]
    return torch.tensor([SOS_INDEX] + ids + [EOS_INDEX], dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, pairs, source_vocabulary, target_vocabulary):
        self.pairs = pairs
        self.source_vocabulary = source_vocabulary
        self.target_vocabulary = target_vocabulary
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        source, target = self.pairs[i]
        return (encode_sentence(source, self.source_vocabulary),
                encode_sentence(target, self.target_vocabulary))

def collate_batch(batch):
    sources, targets = zip(*batch)
    lengths = torch.tensor([len(s) for s in sources], dtype=torch.long)
    return (pad_sequence(sources, batch_first=True, padding_value=PAD_INDEX),
            lengths,
            pad_sequence(targets, batch_first=True, padding_value=PAD_INDEX))

def make_loader(pairs, source_language, target_language, shuffle):
    dataset = TranslationDataset(pairs,
                                 vocabularies[source_language][0],
                                 vocabularies[target_language][0])
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_batch, generator=generator)

print("data utilities ready")

## 2. Model — the attention architecture from Task 4

Task 4 showed that attention beats the fixed-context encoder–decoder by a wide margin
(BLEU 4.16 → 7.68), and that the advantage grows with sentence length. We therefore use
the attention model for every system here. The code is the same as Task 4, restated so
this notebook runs standalone.

In [ ]:
class BahdanauAttention(nn.Module):
    """Additive attention over the encoder states, masked over padding."""

    def __init__(self, hidden_dimension):
        super().__init__()
        self.query_layer = nn.Linear(hidden_dimension, hidden_dimension, bias=False)
        self.key_layer   = nn.Linear(hidden_dimension, hidden_dimension, bias=False)
        self.score_layer = nn.Linear(hidden_dimension, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, source_mask):
        scores = self.score_layer(torch.tanh(
            self.query_layer(decoder_hidden).unsqueeze(1) + self.key_layer(encoder_outputs)
        )).squeeze(-1)
        scores = scores.masked_fill(~source_mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights


class AttentionEncoder(nn.Module):
    def __init__(self, vocabulary_size, embedding_dimension, hidden_dimension):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension,
                                      padding_idx=PAD_INDEX)
        self.lstm = nn.LSTM(embedding_dimension, hidden_dimension, batch_first=True)

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)
        packed = pack_padded_sequence(embedded, source_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_outputs, state = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True,
                                         total_length=source.size(1))
        return outputs, state


class AttentionDecoder(nn.Module):
    def __init__(self, vocabulary_size, embedding_dimension, hidden_dimension):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension,
                                      padding_idx=PAD_INDEX)
        self.attention = BahdanauAttention(hidden_dimension)
        self.lstm = nn.LSTM(embedding_dimension + hidden_dimension, hidden_dimension,
                            batch_first=True)
        self.output_layer = nn.Linear(hidden_dimension * 2, vocabulary_size)

    def forward(self, token, state, encoder_outputs, source_mask):
        embedded = self.embedding(token)
        context, weights = self.attention(state[0][-1], encoder_outputs, source_mask)
        output, state = self.lstm(torch.cat([embedded, context.unsqueeze(1)], dim=-1), state)
        logits = self.output_layer(torch.cat([output.squeeze(1), context], dim=-1))
        return logits, state, weights


class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def forward(self, source, source_lengths, target_input):
        encoder_outputs, state = self.encoder(source, source_lengths)
        source_mask = source != PAD_INDEX
        all_logits = []
        for t in range(target_input.size(1)):
            logits, state, _ = self.decoder(
                target_input[:, t:t + 1], state, encoder_outputs, source_mask)
            all_logits.append(logits)
        return torch.stack(all_logits, dim=1)


def create_model(source_vocabulary_size, target_vocabulary_size,
                 embedding_dimension=64, hidden_dimension=256):
    return AttentionSeq2Seq(
        AttentionEncoder(source_vocabulary_size, embedding_dimension, hidden_dimension),
        AttentionDecoder(target_vocabulary_size, embedding_dimension, hidden_dimension),
    ).to(DEVICE)

print("model defined")

## 3. Training and evaluation

BLEU is smoothed (NLTK `method1`) for the reason given in Task 4: the unsmoothed score
collapses to a meaningless near-zero value whenever a system produces no correct 4-gram.
METEOR uses the appropriate Snowball stemmer per target language, with WordNet synonym
matching disabled (WordNet covers neither Italian nor Swedish).

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.stem import SnowballStemmer
import nltk
for pkg in ("wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

MAX_GRADIENT_NORM = 1.0
MAX_DECODE_LENGTH = 22
SMOOTHING = SmoothingFunction().method1
STEMMERS = {"en": "english", "it": "italian", "sv": "swedish"}


class NoSynonyms:
    @staticmethod
    def synsets(word):
        return []


def run_epoch(model, loader, loss_function, optimizer=None):
    model.train() if optimizer else model.eval()
    total_loss, total_tokens = 0.0, 0
    grad_context = torch.enable_grad() if optimizer else torch.no_grad()
    with grad_context:
        for source, source_lengths, target in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            target_input, target_output = target[:, :-1], target[:, 1:]
            if optimizer:
                optimizer.zero_grad()
            logits = model(source, source_lengths, target_input)
            loss = loss_function(logits.reshape(-1, logits.size(-1)),
                                 target_output.reshape(-1))
            if optimizer:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), MAX_GRADIENT_NORM)
                optimizer.step()
            n_tokens = (target_output != PAD_INDEX).sum().item()
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens
    return total_loss / total_tokens


def train_model(model, name, epochs=12, learning_rate=0.001,
                training_loader=None, validation_loader=None):
    loss_function = nn.CrossEntropyLoss(ignore_index=PAD_INDEX)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_loss, best_state, best_epoch, history = float("inf"), None, -1, []
    started = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, training_loader, loss_function, optimizer)
        val_loss = run_epoch(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        marker = ""
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            marker = "  <- best"
        print(f"  [{name}] epoch {epoch:2d}/{epochs}  train {train_loss:.4f}  "
              f"val {val_loss:.4f}{marker}")

    model.load_state_dict(best_state)
    minutes = (time.time() - started) / 60
    print(f"  [{name}] best epoch {best_epoch} (val {best_loss:.4f}) in {minutes:.1f} min")
    return {"name": name, "best_epoch": best_epoch, "validation_loss": best_loss,
            "training_minutes": minutes, "history": history}


def indices_to_tokens(indices, index_to_token):
    tokens = []
    for i in indices:
        i = int(i)
        if i == EOS_INDEX:
            break
        if i not in (PAD_INDEX, SOS_INDEX):
            tokens.append(index_to_token[i])
    return tokens


@torch.no_grad()
def translate_sentences(model, sentences, source_language, target_language):
    """Greedy-decode a list of raw source sentences into target token lists."""
    model.eval()
    source_vocabulary = vocabularies[source_language][0]
    target_index_to_token = vocabularies[target_language][1]

    outputs = []
    for start in range(0, len(sentences), BATCH_SIZE):
        chunk = sentences[start:start + BATCH_SIZE]
        encoded = [encode_sentence(s, source_vocabulary) for s in chunk]
        lengths = torch.tensor([len(e) for e in encoded], dtype=torch.long)
        source = pad_sequence(encoded, batch_first=True,
                              padding_value=PAD_INDEX).to(DEVICE)

        encoder_outputs, state = model.encoder(source, lengths)
        source_mask = source != PAD_INDEX
        token = torch.full((source.size(0), 1), SOS_INDEX,
                           dtype=torch.long, device=DEVICE)
        finished = torch.zeros(source.size(0), dtype=torch.bool, device=DEVICE)
        generated = []

        for _ in range(MAX_DECODE_LENGTH):
            logits, state, _ = model.decoder(token, state, encoder_outputs, source_mask)
            logits = logits.clone()
            logits[:, [PAD_INDEX, UNK_INDEX, SOS_INDEX]] = float("-inf")
            next_token = logits.argmax(dim=-1)
            next_token = torch.where(finished,
                                     torch.full_like(next_token, EOS_INDEX), next_token)
            generated.append(next_token.cpu())
            finished |= next_token.eq(EOS_INDEX)
            token = next_token.unsqueeze(1)
            if finished.all():
                break

        generated = torch.stack(generated, dim=1)
        outputs.extend(indices_to_tokens(s, target_index_to_token) for s in generated)
    return outputs


def compute_metrics(references, hypotheses, language):
    bleu = 100 * corpus_bleu([[r] for r in references], hypotheses,
                             smoothing_function=SMOOTHING)
    stemmer = SnowballStemmer(STEMMERS[language])
    scores = [meteor_score([r], h, stemmer=stemmer, wordnet=NoSynonyms())
              for r, h in zip(references, hypotheses)]
    return {"BLEU": bleu, "METEOR": 100 * sum(scores) / len(scores)}

print("training utilities ready")

## 4. Train the three systems

Two legs of the pivot (IT→EN, EN→SV) and the direct IT→SV system, all on the same
training split with identical hyperparameters.

> Roughly 15–20 minutes total on a Colab T4.

In [ ]:
EPOCHS = 12
systems = {}

def train_pair(source_language, target_language, source_index, target_index):
    name = f"{source_language}->{target_language}"
    training_loader = make_loader(view(train_triples, source_index, target_index),
                                  source_language, target_language, shuffle=True)
    validation_loader = make_loader(view(val_triples, source_index, target_index),
                                    source_language, target_language, shuffle=False)
    torch.manual_seed(SEED)          # identical initialisation for every system
    model = create_model(len(vocabularies[source_language][0]),
                         len(vocabularies[target_language][0]))
    info = train_model(model, name, epochs=EPOCHS,
                       training_loader=training_loader,
                       validation_loader=validation_loader)
    torch.save(model.state_dict(), MODEL_DIR / f"{source_language}_to_{target_language}.pt")
    systems[name] = {"model": model, "info": info}
    return model

model_it_en = train_pair("it", "en", IT, EN)   # first leg
model_en_sv = train_pair("en", "sv", EN, SV)   # second leg
model_it_sv = train_pair("it", "sv", IT, SV)   # direct, for comparison

## 5. Run the pivot

The pivot is the composition of the two legs: translate Italian to English, then feed
**the model's own English output** — not the reference English — into the second leg.
Using the reference English would leak information the pivot would not have at test time,
and would measure something other than a pivot system.

We also measure that leaked variant deliberately, as an *oracle*: it isolates how much of
the pivot's loss is caused by first-leg errors rather than by the second leg's own
limitations.

In [ ]:
test_italian = [t[IT] for t in test_triples]
test_english  = [t[EN] for t in test_triples]
test_swedish  = [t[SV] for t in test_triples]

reference_english = [tokenize(s) for s in test_english]
reference_swedish = [tokenize(s) for s in test_swedish]

# --- leg 1: IT -> EN ---
hypothesis_english = translate_sentences(model_it_en, test_italian, "it", "en")
metrics_it_en = compute_metrics(reference_english, hypothesis_english, "en")

# --- leg 2 measured alone: reference EN -> SV ---
hypothesis_sv_from_reference = translate_sentences(model_en_sv, test_english, "en", "sv")
metrics_en_sv = compute_metrics(reference_swedish, hypothesis_sv_from_reference, "sv")

# --- the pivot: feed leg 1's OWN output into leg 2 ---
english_bridge = [" ".join(h) if h else "<eos>" for h in hypothesis_english]
hypothesis_pivot = translate_sentences(model_en_sv, english_bridge, "en", "sv")
metrics_pivot = compute_metrics(reference_swedish, hypothesis_pivot, "sv")

# --- direct IT -> SV ---
hypothesis_direct = translate_sentences(model_it_sv, test_italian, "it", "sv")
metrics_direct = compute_metrics(reference_swedish, hypothesis_direct, "sv")

results = pd.DataFrame([
    {"System": "IT->EN (leg 1)",              "Target": "EN", **metrics_it_en},
    {"System": "EN->SV (leg 2, oracle input)","Target": "SV", **metrics_en_sv},
    {"System": "IT->EN->SV (pivot)",          "Target": "SV", **metrics_pivot},
    {"System": "IT->SV (direct)",             "Target": "SV", **metrics_direct},
]).round(2)
results.to_csv(RESULTS_DIR / "pivot_comparison.csv", index=False)
print(results.to_string(index=False))

print(f"\ncost of pivoting vs direct : BLEU {metrics_pivot['BLEU'] - metrics_direct['BLEU']:+.2f}")
print(f"cost of leg-1 errors        : BLEU {metrics_pivot['BLEU'] - metrics_en_sv['BLEU']:+.2f}"
      f"   (pivot vs the same leg-2 model given perfect English)")

## 6. Error propagation

The oracle row above decomposes the pivot's loss. `EN→SV (oracle input)` and
`IT→EN→SV (pivot)` use the *same* second-leg model; the only difference is whether its
input is the reference English or the first leg's output. The drop between them is
therefore attributable entirely to errors inherited from leg 1.

In [ ]:
bars = {
    "leg 2 with\nperfect English": metrics_en_sv["BLEU"],
    "pivot\n(inherited errors)":   metrics_pivot["BLEU"],
    "direct IT->SV":                metrics_direct["BLEU"],
}
fig, ax = plt.subplots(figsize=(6, 3.6))
colours = ["#4C72B0", "#DD8452", "#55A868"]
positions = np.arange(len(bars))
ax.bar(positions, list(bars.values()), 0.6, color=colours)
for x, v in zip(positions, bars.values()):
    ax.text(x, v, f"{v:.1f}", ha="center", va="bottom", fontsize=10)
ax.set_xticks(positions); ax.set_xticklabels(list(bars), fontsize=9)
ax.set_ylabel("BLEU (Swedish output)")
ax.set_title("Cost of pivoting through English")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "pivot_error_propagation.png", dpi=150)
plt.show()

# How much does the first leg degrade before the second one even runs?
print(f"leg 1 (IT->EN) quality      : BLEU {metrics_it_en['BLEU']:.2f}")
print(f"leg 2 given perfect English : BLEU {metrics_en_sv['BLEU']:.2f}")
print(f"pivot end-to-end            : BLEU {metrics_pivot['BLEU']:.2f}")
retained = 100 * metrics_pivot["BLEU"] / metrics_en_sv["BLEU"]
print(f"\npivot retains {retained:.1f}% of the second leg's oracle performance")

In [ ]:
# --- Qualitative examples: watch an error enter at leg 1 and survive to Swedish ---
print("Examples (source -> intermediate English -> Swedish):\n")
for i in range(6):
    print(f"IT  : {test_italian[i]}")
    print(f"EN* : {' '.join(hypothesis_english[i])}      <- leg 1 output (fed to leg 2)")
    print(f"EN  : {test_english[i]}      <- reference English")
    print(f"SV* : {' '.join(hypothesis_pivot[i])}      <- pivot")
    print(f"SV~ : {' '.join(hypothesis_direct[i])}      <- direct")
    print(f"SV  : {test_swedish[i]}      <- reference")
    print()

In [ ]:
# --- Persist everything the report needs ---
summary = {
    "data": {
        "triples_total": len(triples),
        "train": len(train_triples), "validation": len(val_triples),
        "test": len(test_triples),
        "construction": "join it-en and sv-en on the shared English side",
    },
    "hyperparameters": {
        "embedding_dimension": 64, "hidden_dimension": 256,
        "learning_rate": 0.001, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "architecture": "Bahdanau attention seq2seq (Task 4)",
        "decoding": "greedy", "device": str(DEVICE),
    },
    "results": {
        "it_en_leg1": metrics_it_en,
        "en_sv_leg2_oracle": metrics_en_sv,
        "pivot_it_en_sv": metrics_pivot,
        "direct_it_sv": metrics_direct,
    },
    "training_minutes": {k: v["info"]["training_minutes"] for k, v in systems.items()},
    "pivot_cost_bleu_vs_direct": metrics_pivot["BLEU"] - metrics_direct["BLEU"],
    "pivot_cost_bleu_vs_oracle": metrics_pivot["BLEU"] - metrics_en_sv["BLEU"],
}
with open(RESULTS_DIR / "task5_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

for name, entry in systems.items():
    pd.DataFrame(entry["info"]["history"]).to_csv(
        RESULTS_DIR / f"history_{name.replace('->', '_to_')}.csv", index=False)

print(json.dumps(summary["results"], indent=2))
print(f"\nSaved to {RESULTS_DIR}")

In [ ]:
# --- Get the results OFF Colab before the session dies ---
import shutil, sys
archive = shutil.make_archive(str(PROJECT_ROOT / "task5_results"), "zip",
                              root_dir=RESULTS_DIR)
print("archive:", archive)
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name:38} {f.stat().st_size/1024:8.1f} KB")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)
else:
    print("\nNot on Colab - results are already in", RESULTS_DIR)

## 7. What to write in the report

Fill Section~\ref{sec:pivot} with:

1. **Why a pivot is needed** — no Italian–Swedish parallel corpus exists in Europarl,
   while both IT–EN and SV–EN are plentiful.
2. **How the evaluation corpus was built** — joining the two corpora on the shared
   English side, 59,524 triples. This is a genuine contribution and worth a sentence:
   without it the task cannot be measured at all.
3. **The results table** — from `pivot_comparison.csv`.
4. **Error propagation** — the oracle row is the key: the same leg-2 model scores far
   higher on reference English than on leg-1 output, and that difference is exactly the
   inherited error. Use `pivot_error_propagation.png`.
5. **Pivot vs direct** — the direct system sees the same data without an intermediate
   language. If it wins, quantify by how much and say that is the price of pivoting; if
   the pivot is competitive, note that pivoting would let us reach language pairs for
   which no direct data exists at all.
6. **Cost** — the pivot needs two models and two decoding passes at inference.